In [ ]:
#| default_exp main01

## Chat UI (DaisyUI)

Pure [DaisyUI](https://daisyui.com/) — **no MonsterUI / FrankenUI**.
MonsterUI already loads DaisyUI under the hood, so swapping headers alone looks identical.
This notebook uses a different theme (`nord`), navbar, a Chat/Graph taskbar under the input, a viz-picker pill on the message box, and chat avatars so the difference is obvious.

Messages go to the in-process `string_therapy` `/controller`.

In [ ]:
#| export
import uuid
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

from dotenv import load_dotenv
from fasthtml.common import *
from starlette.testclient import TestClient
from string_therapy import create_app

REPO = Path(__file__).resolve().parents[1]
if not (REPO / ".env").exists() and (REPO.parent / ".env").exists():
    REPO = REPO.parent
load_dotenv(REPO / ".env")

_controller = TestClient(create_app())

DAISY_THEME = "nord"

daisy_hdrs = (
    Script(src="https://cdn.tailwindcss.com"),
    Link(
        rel="stylesheet",
        href="https://cdn.jsdelivr.net/npm/daisyui@4.12.22/dist/full.min.css",
        type="text/css",
    ),
    Script(src="https://unpkg.com/lucide@0.468.0"),
    Style("""
      html, body { height: 100%; margin: 0; }
      #main-panel { min-height: 0; }
      .daisy-shell {
        background:
          radial-gradient(ellipse at top, oklch(92% 0.04 230 / 0.9), transparent 55%),
          linear-gradient(180deg, oklch(96% 0.02 240), oklch(98% 0.01 220));
      }
      .chat-bubble { max-width: 36rem; }
    """),
)

In [ ]:
#| export
class IconKind(Enum):
    INTERFACE = "interface"
    IO = "io"

@dataclass
class Icon:
    id: str
    label: str
    icon: str
    kind: IconKind
    active: bool = False
    extra: dict = field(default_factory=dict)

interface_icons = [
    Icon("chat", "Chat", "message-circle", IconKind.INTERFACE, active=True),
    Icon("graph", "Graph", "share-2", IconKind.INTERFACE),
]

io_icons = [
    Icon("scatter", "Scatter", "chart-scatter", IconKind.IO),
    Icon("heatmap", "Heatmap", "grid-3x3", IconKind.IO),
    Icon("timeseries", "Timeseries", "line-chart", IconKind.IO),
    Icon("table", "Table", "table-2", IconKind.IO),
]

GRAPH_HTML = REPO / "ui" / "graph_network.html"
GRAPH_JSON = REPO / "routes" / "router_graph.json"

_VIEW_IDS = (
    "chat-view",
    "graph-view",
    "scatter-view",
    "heatmap-view",
    "timeseries-view",
    "table-view",
)
_ACTIVE_OUTLINE_CLASSES = ("ring-2", "ring-primary", "text-primary")

def lucide_icon(name: str, cls: str = "w-5 h-5"):
    return I(data_lucide=name, cls=cls)

def _show_view_js(view_id: str) -> str:
    parts = [f"document.getElementById('{v}').classList.add('hidden')" for v in _VIEW_IDS]
    parts.append(f"document.getElementById('{view_id}').classList.remove('hidden')")
    return "; ".join(parts)

def _set_active_icon_js(icon_id: str, *, attr: str = "data-taskbar-icon") -> str:
    remove = "".join(f"b.classList.remove('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    add = "".join(f"t.classList.add('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    return (
        f"document.querySelectorAll('[{attr}]').forEach(b=>{{"
        + remove
        + "});"
        + f"const t=document.querySelector('[{attr}=\"{icon_id}\"]');"
        + f"if(t){{{add}}}"
    )

def _view_for_icon(icon_id: str) -> str:
    return {"chat": "chat-view", "graph": "graph-view"}.get(icon_id, "chat-view")

def top_navbar():
    return Div(
        Div(
            A(
                lucide_icon("sparkles", cls="w-5 h-5"),
                Span("String Therapy", cls="font-semibold tracking-tight"),
                cls="btn btn-ghost text-xl gap-2",
            ),
            cls="flex-1",
        ),
        Div(
            Span("DaisyUI · nord", cls="badge badge-primary badge-outline"),
            cls="flex-none",
        ),
        cls="navbar bg-base-100/80 backdrop-blur border-b border-base-300 px-4 shadow-sm",
    )

def _taskbar_btn(ic: Icon):
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        lucide_icon(ic.icon, cls="w-4 h-4"),
        Span(ic.label, cls="text-[10px] leading-tight"),
        type="button",
        title=ic.label,
        id=f"taskbar-icon-{ic.id}",
        data_taskbar_icon=ic.id,
        cls=f"btn btn-ghost btn-sm h-auto py-1.5 px-3 gap-1 flex-col {outline}".strip(),
        onclick="; ".join([
            _set_active_icon_js(ic.id),
            _show_view_js(_view_for_icon(ic.id)),
            "lucide.createIcons();",
        ]),
    )

def _io_icon_btn(ic: Icon, **kw):
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        lucide_icon(ic.icon, cls="w-3.5 h-3.5"),
        title=ic.label,
        id=f"io-icon-{ic.id}",
        data_io_icon=ic.id,
        type="button",
        cls=f"btn btn-ghost btn-circle btn-xs {outline}".strip(),
        **kw,
    )

def render_io_icon(ic: Icon):
    return _io_icon_btn(
        ic,
        onclick="; ".join([
            _set_active_icon_js(ic.id, attr="data-io-icon"),
            _show_view_js(f"{ic.id}-view"),
            "lucide.createIcons();",
        ]),
    )

def taskbar():
    return Div(
        *[_taskbar_btn(ic) for ic in interface_icons],
        id="taskbar",
        cls="flex items-center justify-center gap-1 mt-2 px-1 py-1 rounded-xl bg-base-100/90 border border-base-300 shadow-sm",
    )

def viz_picker():
    return Div(
        *[render_io_icon(i) for i in io_icons],
        id="viz-picker",
        cls=(
            "absolute left-1/2 -translate-x-1/2 -top-3 z-10 "
            "inline-flex items-center gap-0.5 px-1.5 py-0.5 "
            "rounded-full border border-base-300 bg-base-100 shadow-sm"
        ),
    )

def chat_bubble(role: str, content: str):
    is_user = role == "user"
    avatar = Div(
        Div(
            lucide_icon("user" if is_user else "bot", cls="w-4 h-4"),
            cls="w-10 rounded-full bg-primary text-primary-content grid place-items-center"
            if is_user
            else "w-10 rounded-full bg-secondary text-secondary-content grid place-items-center",
        ),
        cls="chat-image avatar",
    )
    return Div(
        avatar,
        Div("You" if is_user else "Router", cls="chat-header opacity-70 text-xs mb-1"),
        Div(
            content,
            cls=f"chat-bubble {'chat-bubble-primary' if is_user else 'chat-bubble-secondary'} whitespace-pre-line shadow",
        ),
        cls=f"chat {'chat-end' if is_user else 'chat-start'} px-2",
    )

def oob_chat_bubble(role: str, content: str):
    return Div(
        chat_bubble(role, content),
        Script("lucide.createIcons();"),
        hx_swap_oob="beforeend:#chat-messages",
    )

def chat_area():
    return Div(
        Div(
            Div(
                Span("Ask the materials router", cls="font-medium"),
                P("Routes over Postgres + Apache AGE", cls="text-sm opacity-60"),
                cls="mb-4 px-2",
            ),
            Div(id="chat-messages", cls="flex flex-col gap-3 w-full"),
            cls="w-full max-w-2xl mx-auto py-6 px-3",
        ),
        id="chat-view",
        cls="flex flex-col flex-1 min-h-0 overflow-y-auto",
    )

def graph_view():
    return Div(
        Div(
            Span("Router graph", cls="font-medium"),
            Span("live", cls="badge badge-accent badge-sm ml-2"),
            cls="px-4 py-2 border-b border-base-300 bg-base-100/70",
        ),
        Iframe(src="/graph", title="Router graph", cls="w-full flex-1 border-0 bg-base-100"),
        id="graph-view",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden hidden",
    )

def _viz_placeholder(ic: Icon):
    return Div(
        Div(
            lucide_icon(ic.icon, cls="w-12 h-12 opacity-30"),
            P(f"{ic.label} visualization", cls="text-sm opacity-60 mt-3"),
            cls="flex flex-col flex-1 items-center justify-center",
        ),
        id=f"{ic.id}-view",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden hidden",
    )

def main_panel():
    return Div(
        chat_area(),
        graph_view(),
        *[_viz_placeholder(i) for i in io_icons],
        id="main-panel",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden pb-40",
    )

def input_bar():
    return Div(
        Form(
            Div(
                viz_picker(),
                Textarea(
                    placeholder="Message the router…",
                    name="message",
                    rows=2,
                    cls="textarea textarea-bordered textarea-primary w-full resize-none rounded-2xl pt-6 pr-14 leading-snug",
                    id="chat-input",
                    onkeydown="if(event.key==='Enter'&&!event.shiftKey){event.preventDefault();this.form.requestSubmit()}",
                    hx_on_input="""
                        let btn = document.getElementById('submit-btn');
                        if (this.value.trim().length > 0) btn.classList.remove('btn-disabled');
                        else btn.classList.add('btn-disabled');
                        """,
                ),
                Button(
                    lucide_icon("send", cls="w-4 h-4"),
                    id="submit-btn",
                    cls="btn btn-primary btn-circle btn-sm absolute right-3 bottom-3 btn-disabled",
                    type="submit",
                ),
                cls="relative w-full",
            ),
            hx_post="/chat/send",
            hx_target="#htmx-sink",
            hx_swap="none",
            hx_on__after_request=(
                "document.getElementById('chat-input').value='';"
                "document.getElementById('submit-btn').classList.add('btn-disabled');"
                "lucide.createIcons();"
            ),
            cls="w-full",
        ),
        taskbar(),
        id="input-bar",
        cls="fixed bottom-0 left-0 right-0 z-10 px-3 pb-3 pt-5 max-w-2xl mx-auto w-full",
    )

app, rt = fast_app(
    hdrs=daisy_hdrs,
    pico=False,
    htmlkw={"data-theme": DAISY_THEME},
    bodykw={"class": "daisy-shell min-h-screen"},
)

@rt("/graph")
def graph():
    import json as _json
    html = GRAPH_HTML.read_text(encoding="utf-8")
    data = _json.loads(GRAPH_JSON.read_text(encoding="utf-8"))
    marker = '<script id="graph-data" type="application/json">null</script>'
    payload = f'<script id="graph-data" type="application/json">{_json.dumps(data)}</script>'
    if marker not in html:
        raise RuntimeError("graph_network.html missing #graph-data injection point")
    return Response(html.replace(marker, payload, 1), media_type="text/html")

@rt("/")
def get(session):
    if "session_id" not in session:
        session["session_id"] = str(uuid.uuid4())

    return Div(
        top_navbar(),
        main_panel(),
        input_bar(),
        Div(id="htmx-sink", cls="hidden", aria_hidden="true"),
        Script("lucide.createIcons();"),
        cls="flex flex-col h-screen",
    )

@rt("/chat/send", methods=["POST"])
async def chat_send(message: str, session):
    user_msg = message.strip()
    if not user_msg:
        return ""

    sid = session.get("session_id", "").strip()
    payload: dict = {"message": user_msg, "use_history": True, "response_format": "json"}
    if sid:
        payload["session_id"] = sid

    r = _controller.post("/controller", json=payload)
    try:
        data = r.json()
    except Exception:  # noqa: BLE001
        data = {"detail": r.text[:2000]}

    if r.status_code >= 400:
        reply = data.get("detail") or data.get("error") or f"Controller error HTTP {r.status_code}"
    else:
        reply = data.get("response") or data.get("detail") or "No response"
        if not isinstance(reply, str):
            import json as _json
            reply = _json.dumps(reply, indent=2)

    new_sid = (data.get("session_id") or sid or "").strip()
    if new_sid:
        session["session_id"] = new_sid

    return (
        oob_chat_bubble("user", user_msg),
        oob_chat_bubble("assistant", reply),
        Script(
            "requestAnimationFrame(()=>{const c=document.getElementById('chat-view');"
            "if(c)c.scrollTop=c.scrollHeight; lucide.createIcons();})"
        ),
    )


if __name__ == "__main__":
    serve(appname="string_therapy_for_material_science.main01", port=4546)

## One-time DB setup (optional)

Run when you need to (re)bootstrap schema + seed the router graph. Not exported into `main01`.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from string_therapy import ensure_schema, seed_router_db

repo = Path.cwd() if (Path.cwd() / '.env').exists() else Path.cwd().parent
load_dotenv(repo / '.env')

ensure_schema()
seed_router_db(repo / 'routes' / 'router_graph.json')
print('schema + seed ok')